# 콘텐츠 기반 필터링 추천 시스템
- 콘텐츠 기반 추천 시스템은 **사용자가 이전에 좋아했던 아이템의 특징(콘텐츠)** 을 분석하여 비슷한 아이템을 추천하는 시스템
- 예를 들어, 넷플릭스에서 사용자가 로맨틱 코미디 영화를 좋아하면, 비슷한 로맨틱 코미디 영화를 추천하는 방식

# 1.라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 2.데이터셋 불러오기
- TMDB 5000 영화 데이터 셋 : 유명한 영화 데이터 정보 사이트인 IMDB의 많은 영화 중 주요 5000개 영화에 대한 메타 정보를 새롭게 가공해 캐글에 제공한 데이터 셋
- https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

In [2]:
movie_df = pd.read_csv('../data1/tmdb_5000_movies.csv')
movie_df.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [3]:
movie_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [4]:
movie_df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [5]:
# 필요한 컬럼만 추출
movie_df = movie_df[['id', 'title', 'genres', 'vote_average', "vote_count", 'popularity', 'keywords', 'overview']]

In [6]:
movie_df.head()

,id,title,genres,vote_average,vote_count,popularity,keywords,overview
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",7.2,11800,150.437577,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",6.9,4500,139.082615,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",6.3,4466,107.376788,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",7.6,9106,112.312950,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",Following the death of District Attorney Harve...
4,49529,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",6.1,2124,43.926995,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","John Carter is a war-weary, former military ca..."


In [7]:
# 최대 400자까지 보겠다.
pd.set_option('max_colwidth', 400)
movie_df[['genres', 'keywords']][:1]

,genres,keywords
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""Science Fiction""}]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 3388, ""name"": ""space colony""}, {""id"": 3679, ""name"": ""society""}, {""id"": 3801, ""name"": ""space travel""}, {""id"": 9685, ""name"": ""futuristic""}, {""id"": 9840, ""name"": ""romance""}, {""id"": 9882, ""name"": ""space""}, {""id"": 9951, ""name"": ""alien""}, {""id"": 10148, ""name"": ""tribe""}, {""id"": 10158, ""na..."


In [8]:
# 문자열로 저장된 리스트 형식을 실제 리스트로 변환하기 위해 사용
from ast import literal_eval

In [9]:
movie_df['genres'] = movie_df['genres'].apply(literal_eval)
movie_df['keywords'] = movie_df['keywords'].apply(literal_eval)

In [10]:
movie_df[['genres', 'keywords']].iloc[0]

genres                                                                                                                                                                                                                                                                               [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]
keywords    [{'id': 1463, 'name': 'culture clash'}, {'id': 2964, 'name': 'future'}, {'id': 3386, 'name': 'space war'}, {'id': 3388, 'name': 'space colony'}, {'id': 3679, 'name': 'society'}, {'id': 3801, 'name': 'space travel'}, {'id': 9685, 'name': 'futuristic'}, {'id': 9840, 'name': 'romance'}, {'id': 9882, 'name': 'space'}, {'id': 9951, 'name': 'alien'}, {'id': 10148, 'name': 'tribe'}, {'id': 10158, 'na...
Name: 0, dtype: object

In [11]:
genres = []         # 모든 영화의 장르 이름을 담을 리스트

for tmp_genres in movie_df['genres']:
    tmpList = []        # 각 영화의 장르 이름을 임시로 저장
    for i in tmp_genres:
        tmpList.append(i['name'])    # name 키에 해당하는 값을 추출
    genres.append(tmpList)

In [12]:
genres

[['Action', 'Adventure', 'Fantasy', 'Science Fiction'],
 ['Adventure', 'Fantasy', 'Action'],
 ['Action', 'Adventure', 'Crime'],
 ['Action', 'Crime', 'Drama', 'Thriller'],
 ['Action', 'Adventure', 'Science Fiction'],
 ['Fantasy', 'Action', 'Adventure'],
 ['Animation', 'Family'],
 ['Action', 'Adventure', 'Science Fiction'],
 ['Adventure', 'Fantasy', 'Family'],
 ['Action', 'Adventure', 'Fantasy'],
 ['Adventure', 'Fantasy', 'Action', 'Science Fiction'],
 ['Adventure', 'Action', 'Thriller', 'Crime'],
 ['Adventure', 'Fantasy', 'Action'],
 ['Action', 'Adventure', 'Western'],
 ['Action', 'Adventure', 'Fantasy', 'Science Fiction'],
 ['Adventure', 'Family', 'Fantasy'],
 ['Science Fiction', 'Action', 'Adventure'],
 ['Adventure', 'Action', 'Fantasy'],
 ['Action', 'Comedy', 'Science Fiction'],
 ['Action', 'Adventure', 'Fantasy'],
 ['Action', 'Adventure', 'Fantasy'],
 ['Action', 'Adventure'],
 ['Adventure', 'Fantasy'],
 ['Adventure', 'Fantasy'],
 ['Adventure', 'Drama', 'Action'],
 ['Drama', 'Romance

In [13]:
movie_df['genres'] = genres
movie_df.head(3)

,id,title,genres,vote_average,vote_count,popularity,keywords,overview
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]",7.2,11800,150.437577,"[{'id': 1463, 'name': 'culture clash'}, {'id': 2964, 'name': 'future'}, {'id': 3386, 'name': 'space war'}, {'id': 3388, 'name': 'space colony'}, {'id': 3679, 'name': 'society'}, {'id': 3801, 'name': 'space travel'}, {'id': 9685, 'name': 'futuristic'}, {'id': 9840, 'name': 'romance'}, {'id': 9882, 'name': 'space'}, {'id': 9951, 'name': 'alien'}, {'id': 10148, 'name': 'tribe'}, {'id': 10158, 'na...","In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization."
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]",6.9,4500,139.082615,"[{'id': 270, 'name': 'ocean'}, {'id': 726, 'name': 'drug abuse'}, {'id': 911, 'name': 'exotic island'}, {'id': 1319, 'name': 'east india trading company'}, {'id': 2038, 'name': 'love of one's life'}, {'id': 2052, 'name': 'traitor'}, {'id': 2580, 'name': 'shipwreck'}, {'id': 2660, 'name': 'strong woman'}, {'id': 3799, 'name': 'ship'}, {'id': 5740, 'name': 'alliance'}, {'id': 5941, 'name': 'caly...","Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems."
2,206647,Spectre,"[Action, Adventure, Crime]",6.3,4466,107.376788,"[{'id': 470, 'name': 'spy'}, {'id': 818, 'name': 'based on novel'}, {'id': 4289, 'name': 'secret agent'}, {'id': 9663, 'name': 'sequel'}, {'id': 14555, 'name': 'mi6'}, {'id': 156095, 'name': 'british secret service'}, {'id': 158431, 'name': 'united kingdom'}]","A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit to reveal the terrible truth behind SPECTRE."


In [13]:
# keywords 컬럼도 각 영화의 키워드 이름만 추출
movie_df['keywords'] = movie_df['keywords'].apply(lambda x: [i['name'] for i in x])

movie_df.head()

,id,title,genres,vote_average,vote_count,popularity,keywords,overview
0,19995,Avatar,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]",7.2,11800,150.437577,"[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldier, battle, love affair, anti war, power relations, mind and soul, 3d]","In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization."
1,285,Pirates of the Caribbean: At World's End,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 28, 'name': 'Action'}]",6.9,4500,139.082615,"[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afterlife, fighter, pirate, swashbuckler, aftercreditsstinger]","Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems."
2,206647,Spectre,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 80, 'name': 'Crime'}]",6.3,4466,107.376788,"[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit to reveal the terrible truth behind SPECTRE."
3,49026,The Dark Knight Rises,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'name': 'Crime'}, {'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}]",7.6,9106,112.312950,"[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess, tragic hero, terrorism, destruction, catwoman, cat burglar, imax, flood, criminal underworld, batman]","Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation and is subsequently hunted by the Gotham City Police Department. Eight years later, Batman encounters the mysterious Selina Kyle and the villainous Bane, a new terrorist leader who overwhelms Gotham's finest. The Dark Knight resurfaces to protect a cit..."
4,49529,John Carter,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 878, 'name': 'Science Fiction'}]",6.1,2124,43.926995,"[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, alien race, superhuman strength, mars civilization, sword and planet, 19th century, 3d]","John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic planet of Barsoom (Mars) and reluctantly becomes embroiled in an epic conflict. It's a world on the brink of collapse, and Carter rediscovers his humanity when he realizes the survival of Barsoom and its people rests in his hands."


In [15]:
# keywords, genres 컬럼의 리스트 데이터를 공백을 기준으로 하나의 문자열로 변환
movie_df['genres'] = movie_df['genres'].apply(lambda x: ' '.join([i['name'] for i in x] if x and isinstance(x[0], dict) else x))
movie_df['keywords'] = movie_df['keywords'].apply(lambda x: ' '.join([i['name'] for i in x] if x and isinstance(x[0], dict) else x))

In [16]:
movie_df.head(1)

,id,title,genres,vote_average,vote_count,popularity,keywords,overview
0,19995,Avatar,Action Adventure Fantasy Science Fiction,7.2,11800,150.437577,culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization."


# 3.장르 콘텐츠 유사도 측정
- 문자열로 변환된 genres 칼럼을 Count 기반으로 피처 벡터화 변환
- genres 문자열을 피처 벡터화 행렬로 변환한 데이터 세트를 코사인 유사도를 통해 비교
- 이를 위해 데이터 세트의 레코드별로 타 레코드와 장르에서 코사인 유사도 값을 가지는 객체를 생성
- 장르 유사도가 높은 영화 중에 평점이 높은 순으로 영화를 추천

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

# genres 컬럼을 수치 벡터로 변환. 문장에서 등장하는 단어의 빈도를 기반으로 행렬을 생성
count_vect = CountVectorizer(min_df=1,          # 최소 한번이라도 등장한 단어만 벡터 포함
                             ngram_range=(1,2)) # 단어1, 단어 2개까지 고려

genres_mat = count_vect.fit_transform(movie_df['genres'])

In [18]:
genres_mat.shape

(4803, 276)

- 코사인유사도
    - 벡터 간의 각도를 기반으로 텍스트나 수치 데이터의 유사성을 측정
    - 값은 **0에서 1 사이** 이며, **1에 가까울수록 유사**
    - 코사인 유사도의 일반적인 범위는 -1~1 이지만 추천시스템에서 자주 쓰는 TF-IDF, BoW, 빈도 기반 벡터처럼 값이 음수가 아닌 경우에는 보통 범위가 0~1 이다.
    - 장르와 키워드는 빈도 기반 데이터이기 때문에 코사인 유사도를 사용하는 것이 적합

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
# 장르 벡터 행렬 사이의 코사인 유사도 계산
# 모든 영화 장르를 벡터화 한 후 자기 자신과 비교해서 영화 간 유사도 행렬 생성
genres_sim = cosine_similarity(genres_mat, genres_mat)

# genres_mat[:2]

genres_sim.argsort()[:,::-1]   # 내림차순 정렬, 유사도가 높은 순으로 정렬

array([[   0,   46, 3494, ..., 3600, 3569, 4799],
       [   1,  129,   12, ..., 4754, 4753, 3829],
       [1740, 1542,    2, ..., 4757, 4758, 4795],
       ...,
       [4800, 3809,  323, ..., 4771, 3640,   41],
       [4802, 4801, 4800, ...,    2,    1,    0],
       [4698, 4524, 3891, ...,    2,    1, 4800]], shape=(4803, 4803))

In [21]:
# 유사도 행렬을 정렬해 각 영화와 가장 비슷한 영화의 인덱스 순서를 만든다.
# artsort 결과를 역순으로 뒤집어서 유사도가 높은 항목부터 조회할 수 있게

genres_sim_sorted_ind = genres_sim.argsort()[:, ::-1]
genres_sim_sorted_ind[:1]

array([[   0,   46, 3494, ..., 3600, 3569, 4799]], shape=(1, 4803))

# 4.장르 콘텐츠 필터링을 이용한 영화 추천

In [22]:
# 유사 영화 찾기 함수(입력된 영화와 유사도가 높은 영화를 찾는다.)
def find_sim_movie(df, sorted_ind, title_name, top_n=10):

    # 입력된 영화 제목으로 해당 영화의 데이터와 인덱스를 찾음
    title_movie = df[df['title'] == title_name]

    # 해당 영화의 인덱스 값을 가져옴
    title_index = title_movie.index.values

    # 해당 영화와 유사한 영화들의 인덱스를 가져옴
    similar_indexes = sorted_ind[title_index, :top_n]
    print(similar_indexes)      # 확인
    similar_indexes = similar_indexes.reshape(-1)

    # 해당 인덱스에 해당하는 영화 리턴
    return df.iloc[similar_indexes]

In [23]:
similar_movies = find_sim_movie(movie_df, genres_sim_sorted_ind, 'The Godfather', 10)       # The Godfather와 가장 유사한 영화 10편 찾아줘.
similar_movies[['title', 'vote_average']]

[[1881 3378 3866 1370 1464  588 3887 3594 2839  892]]


,title,vote_average
1881,The Shawshank Redemption,8.5
3378,Auto Focus,6.1
3866,City of God,8.1
1370,21,6.5
1464,Black Water Transit,0.0
588,Wall Street: Money Never Sleeps,5.8
3887,Trainspotting,7.8
3594,Spring Breakers,5.0
2839,Rounders,6.9
892,Casino,7.8


- 평점 가중치 기반 영화 추천



In [24]:
percetile = 0.6         # 상위 60% 기준
m = movie_df['vote_count'].quantile(percetile)      # 370 : 평가 수가 370개 이상인 영화만 신뢰할 수 있는 영화로 간주
c = movie_df['vote_average'].quantile(percetile)    # 6.5 : 평균 평점이 6.5 이상인 영화들을 추천 대상으로 고려
m, c

(np.float64(370.1999999999998), np.float64(6.5))

In [25]:
def weight_vote_average(record):
    v = record['vote_count']        # 특정 영화의 평가수
    r = record['vote_average']      # 특정 영화의 평균 평점

    # IMDB에서 평가 횟수에 대한 가중치가 부여된 평점 방식
    score = ((v/(v+m))*r) + ((m/(m+v))*c)

    return score

In [26]:
# 가중 평점을 컬럼에 저장
movie_df['weight_vote_average'] = movie_df.apply(weight_vote_average, axis=1)

In [27]:
movie_df[['title', 'vote_average', 'weight_vote_average', 'vote_count']].sort_values(by = 'weight_vote_average', ascending=False)[:5]

,title,vote_average,weight_vote_average,vote_count
1881,The Shawshank Redemption,8.5,8.413658,8205
3337,The Godfather,8.4,8.287696,5893
662,Fight Club,8.3,8.231887,9413
3232,Pulp Fiction,8.3,8.224262,8428
1818,Schindler's List,8.3,8.158197,4329


In [28]:
# 가중 평점 기반 영화 추천
def find_sim_movie(df, sorted_ind, title_name, top_n=10):
    title_movie = df[df['title'] == title_name]
    title_index = title_movie.index.values

    # 유사도가 높은 영화 인덱스 추출(top_n의 두배에 해당되는)
    similar_indexes = sorted_ind[title_index, :(top_n*2)]
    similar_indexes = similar_indexes.reshape(-1)

    # 기존 영화 index 제외
    similar_indexes = similar_indexes[similar_indexes != title_index]

    return df.iloc[similar_indexes].sort_values('weight_vote_average', ascending = False)[:top_n]

In [29]:
find_sim_movies = find_sim_movie(movie_df, genres_sim_sorted_ind, 'Avatar', 10)
find_sim_movies[['title', 'vote_average', 'weight_vote_average']]

,title,vote_average,weight_vote_average
46,X-Men: Days of Future Past,7.5,7.442176
813,Superman,6.9,6.793636
3208,Star Wars: Clone Wars: Volume 1,8.0,6.601964
420,Hellboy II: The Golden Army,6.5,6.500000
870,Superman II,6.5,6.500000
14,Man of Steel,6.5,6.500000
3494,Beastmaster 2: Through the Portal of Time,4.6,6.416581
1932,Sheena,5.0,6.415859
1191,Small Soldiers,6.2,6.326033
232,The Wolverine,6.3,6.316739
